# Nettoyage des données du Congrès (2020-2026) — donnée propre pour backtest

**Objectif.** Partir de notre donnée validée 2020-2026 et produire, *étape par étape*, un jeu de données
**propre et prêt pour un backtest**. Chaque filtre est **justifié** et **chiffré** (entonnoir).

**Principe.**
- **Source** : `common.quality.load_final` — notre panel *golden* 2020-2026 (89 852 transactions uniques),
  déjà dédupliqué et corrigé (dates + tickers) à la lecture. *Quiver n'est jamais réinjecté* ; on n'utilise
  **pas** la table hybride 2014-2026 (qui, elle, contient du Quiver).
- **Nettoyage strict** : à chaque étape on **retire** les lignes non-backtestables (on ne les garde pas avec
  un simple drapeau). Périmètre retenu : **actions cotées + ETF uniquement**.
- **Réutilisation** : tous les filtres s'appuient sur des fonctions **déjà écrites et testées** du dépôt —
  on ne réécrit aucune logique de nettoyage.

**Sortie** : `data/clean/transactions_backtest_2020_2026.csv`.

In [1]:
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

# --- localisation de la racine du dépôt (contient common/ et data/) ---
def _find_repo():
    for c in [Path.cwd(), *Path.cwd().parents]:
        if (c / "common" / "quality.py").exists() and (c / "data" / "house" / "tables").exists():
            return c
    raise RuntimeError("Racine du dépôt introuvable")

REPO = _find_repo()
sys.path.insert(0, str(REPO))

# --- fonctions réutilisées (aucune logique de nettoyage réécrite ici) ---
from common.quality import load_final, _asset_bucket    # chargement + famille d'actif
from house.quiver import norm_ticker                     # normalisation du ticker
from common.quiver_diagnosis import _quiver_untradeable  # ticker non coté (CUSIP, $, fragment OCR…)

print("Dépôt :", REPO)

# --- petit utilitaire pour tracer l'entonnoir (étape, filtre, retirées, restantes) ---
funnel = []
def _step(code, label, before, after):
    funnel.append({"étape": code, "filtre": label,
                   "retirées": before - after, "restantes": after})
    print(f"[{code}] {label}\n    {before:,} → {after:,}   (retirées : {before - after:,})")

Dépôt : /Users/lemairealice/Downloads/Jupiter


## Chargement — notre donnée validée 2020-2026

`load_final` assemble les 4 sous-corpus (House / Sénat × électronique / OCR), déduplique les re-divulgations
d'une année sur l'autre, applique les corrections *read-time* (dates, tickers) et dérive les colonnes utiles
(`lag_days`, `op`, `amount_midpoint`, `corpus`…). On n'a donc rien à recalculer en amont.

In [2]:
df = load_final(REPO)
N0 = len(df)
print(f"{N0:,} transactions uniques chargées\n")
print("Répartition par sous-corpus :")
print(df["corpus"].value_counts().to_string())

89,852 transactions uniques chargées

Répartition par sous-corpus :
corpus
House OCR             48940
House électronique    32667
Sénat électronique     6566
Sénat OCR              1679


## Étape A — Dates présentes et cohérentes

Un backtest a besoin d'une **chronologie fiable**. On retire les lignes où :
- une date (transaction *ou* divulgation) est **illisible / absente** → `lag_days` vaut `NaN` (on ne sait pas
  *quand* agir) ;
- la **divulgation précède la transaction** (`lag_days < 0`) : c'est **impossible** (on « saurait » avant que
  le trade existe) — ce sont des coquilles du déposant ou des erreurs d'OCR ;
- garde-fou : l'**année de transaction** est hors plage plausible (avant 2012 ou après l'année de dépôt).

`lag_days` (= divulgation − transaction, en jours) est **déjà calculé** par `load_final`.

In [3]:
n = len(df)
fy = pd.to_numeric(df["file_year"], errors="coerce")
mask_parse    = df["lag_days"].notna()                        # dates lisibles
mask_coherent = df["lag_days"] >= 0                           # divulgation ≥ transaction
mask_year     = (df["txn_year"] >= 2012) & (df["txn_year"] <= fy)   # année plausible
keep = mask_parse & mask_coherent & mask_year

# aperçu : quelques dates incohérentes retirées (divulgation AVANT transaction)
apercu = (df[mask_parse & (df["lag_days"] < 0)]
          [["declarant_name", "ticker", "transaction_date", "disclosure_date", "lag_days"]].head(5))
print("Exemples de dates incohérentes retirées (divulgation avant transaction) :")
print(apercu.to_string(index=False), "\n")

df = df[keep].copy()
_step("A", "dates présentes & cohérentes", n, len(df))

Exemples de dates incohérentes retirées (divulgation avant transaction) :
       declarant_name ticker transaction_date disclosure_date  lag_days
       Adam Kinzinger   USFD       2020-07-23      2020-07-21      -2.0
       John B. Larson   SBUX       2020-12-18      2020-01-31    -322.0
Donald Sternoff Beyer    NaN       2020-12-26      2020-01-02    -359.0
Donald Sternoff Beyer  BRK.B       2020-12-24      2020-01-02    -357.0
         Rohit Khanna    PEG       2020-07-21      2020-07-17      -4.0 

[A] dates présentes & cohérentes
    89,852 → 89,511   (retirées : 341)


## Étape B — Point-in-time : la date d'entrée = la divulgation

**Le point clé pour un backtest.** L'information n'est **exploitable qu'au moment où elle devient publique**,
c'est-à-dire à la **date de divulgation** — et non à la date de transaction (connue seulement du membre).
On matérialise donc la date d'entrée dans une colonne **`signal_date = disclosure_date`**, pour que le backtest
n'ait jamais à la re-dériver. *Aucune ligne n'est retirée à cette étape.*

In [4]:
n = len(df)
df["signal_date"] = df["disclosure_date"]   # date d'entrée exploitable (point-in-time)
_step("B", "point-in-time (signal_date = disclosure_date)", n, len(df))

[B] point-in-time (signal_date = disclosure_date)
    89,511 → 89,511   (retirées : 0)


## Étape C — Actions cotées et ETF uniquement

Un backtest a besoin d'un **prix**, donc d'un **ticker coté valide**. On garde une ligne si :
- son ticker se **normalise en un symbole non vide** (`norm_ticker`), **et**
- ce ticker est **réellement coté** (on écarte via `_quiver_untradeable` les CUSIP, préférentielles `$`,
  fragments d'OCR, échéances obligataires…), **et**
- sa **famille d'actif n'est pas explicitement non-cotée** (`_asset_bucket` ∉ {obligation, muni, gouvernement,
  option, « autre »}).

**On raisonne « ticker d'abord ».** Une ligne au ticker coté valide est gardée *même si son `asset_type` est
vide* — cas fréquent en House OCR : des milliers de vraies actions (NVDA, IBM, ASML…) sans étiquette de type.
Un filtre « type d'abord » les jetterait à tort (≈ 3 000 lignes).

In [5]:
n = len(df)
NON_COTE = {"bond", "muni", "gov", "option", "autre"}
mask_ticker   = df["ticker"].map(norm_ticker) != ""                    # symbole non vide
mask_tradable = ~df["ticker"].map(_quiver_untradeable)                 # réellement coté
mask_famille  = ~df["asset_type"].map(_asset_bucket).isin(NON_COTE)    # pas une famille non-cotée
keep = mask_ticker & mask_tradable & mask_famille

apercu = df[~keep][["declarant_name", "asset_description", "asset_type", "ticker"]].head(5)
print("Exemples de lignes retirées (non cotées / non tickérisées) :")
print(apercu.to_string(index=False), "\n")

df = df[keep].copy()
_step("C", "actions + ETF cotés (ticker-first)", n, len(df))

Exemples de lignes retirées (non cotées / non tickérisées) :
            declarant_name                                         asset_description asset_type ticker
             Michael Waltz                               Metis Solutions Corporation         PS    NaN
             Michael Waltz         Metis Solutions Corporation - Options  $1,000,000         PS    NaN
            Greg Gianforte                          NTT dOCOMO INC SPON AdR  $50,000      Other    NaN
Neal Patrick Dunn MD, FACS    BancorpSouth Bank 5.50% Series Preferred Stock (BXS$A)      Stock    NaN
Neal Patrick Dunn MD, FACS J P Morgan Chase & Co Depositary Stock, Series AA (JPM$g)      Stock    NaN 

[C] actions + ETF cotés (ticker-first)
    89,511 → 73,487   (retirées : 16,024)


## Étape D — Direction exploitable (achat / vente)

Pour simuler une position, il faut un **sens clair**. On garde les **achats** et les **ventes** ; on retire les
`exchange` et les rares `other`, dont l'économie est ambiguë. *On ne présume pas la stratégie* : conserver les
deux sens laisse le backtest décider (long / short, étude d'événement, sorties). Le sens est déjà normalisé
dans la colonne `op` par `load_final`.

In [6]:
n = len(df)
df = df[df["op"].isin(["buy", "sell"])].copy()
_step("D", "direction ∈ {achat, vente}", n, len(df))

[D] direction ∈ {achat, vente}
    73,487 → 73,016   (retirées : 471)


## Étape E — Montant présent

Pour **dimensionner** une position (en notionnel), il faut un montant. On retire les lignes sans
`amount_midpoint` numérique (le point milieu de la fourchette déclarée, déjà calculé par `load_final`).
Coût négligeable (< 1 %).

In [7]:
n = len(df)
df = df[df["amount_midpoint"].notna()].copy()
_step("E", "montant présent (amount_midpoint)", n, len(df))

[E] montant présent (amount_midpoint)
    73,016 → 72,368   (retirées : 648)


## Récapitulatif de l'entonnoir

In [8]:
rows = [{"étape": "—", "filtre": "départ (load_final)", "retirées": 0, "restantes": N0}] + funnel
recap = pd.DataFrame(rows)
print(recap.to_string(index=False))
print(f"\nDonnée propre finale : {len(df):,} lignes  ({100 * len(df) / N0:.1f} % du panel de départ)\n")

print("Par chambre :");     print(df["chamber"].value_counts().to_string())
print("\nPar sous-corpus :"); print(df["corpus"].value_counts().to_string())
print("\nPar sens :");        print(df["op"].value_counts().to_string())

étape                                        filtre  retirées  restantes
    —                           départ (load_final)         0      89852
    A                  dates présentes & cohérentes       341      89511
    B point-in-time (signal_date = disclosure_date)         0      89511
    C            actions + ETF cotés (ticker-first)     16024      73487
    D                    direction ∈ {achat, vente}       471      73016
    E             montant présent (amount_midpoint)       648      72368

Donnée propre finale : 72,368 lignes  (80.5 % du panel de départ)

Par chambre :
chamber
house     67706
senate     4662

Par sous-corpus :
corpus
House OCR             39818
House électronique    27888
Sénat électronique     4247
Sénat OCR               415

Par sens :
op
sell    36456
buy     35912


## Export — `data/clean/transactions_backtest_2020_2026.csv`

19 colonnes en `snake_case`, prêtes pour le backtest : **identité du membre**, **caractéristiques du trade**,
les **trois dates** (dont `signal_date` = date d'entrée point-in-time) et des colonnes de **traçabilité**
(`doc_id`, `provenance`, `natural_key_hash`).

In [9]:
OUT_DIR = REPO / "data" / "clean"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT = OUT_DIR / "transactions_backtest_2020_2026.csv"

COLS = ["bioguide_id", "member_name", "party", "chamber", "state_district", "committee_membership",
        "ticker", "asset_type", "sector_gics", "etf_proxy", "direction", "amount_midpoint",
        "amount_range", "transaction_date", "disclosure_date", "signal_date",
        "doc_id", "provenance", "natural_key_hash"]

export = (df.rename(columns={"declarant_name": "member_name", "op": "direction"})
            .reindex(columns=COLS))
export.to_csv(OUT, index=False)

print(f"Écrit : {OUT}")
print(f"{len(export):,} lignes × {export.shape[1]} colonnes\n")
print(export.head().to_string(index=False))

Écrit : /Users/lemairealice/Downloads/Jupiter/data/clean/transactions_backtest_2020_2026.csv
72,368 lignes × 19 colonnes

bioguide_id  member_name      party chamber state_district                                                                                                                                                                                                  committee_membership ticker asset_type            sector_gics etf_proxy direction  amount_midpoint     amount_range transaction_date disclosure_date signal_date   doc_id           provenance                                                 natural_key_hash
    S001189 Austin Scott Republican   house           GA08 HLIG01; HLIG02; HLIG06; HSAG03; HSAG16; HSAG22; HSAS03; HSAS26; HSRU04; House Committee on Agriculture; House Committee on Armed Services; House Committee on Rules; House Permanent Select Committee on Intelligence   PLUG      Stock            Industrials       XLI      sell           8000.5 $1,001 - $15,000   